Tyes of retrievers
based on data source:
> Wikipedia retriever - Query the qikipedia API to fetch relevant content for a given query. Some keyword based matching is done under the hood.

> Vector retriever: Most common type of retriever, that searcha nd fetch doc from a vector store based on semantic similarity using vector embeddings.
> 
Based on search strategy
> MMR: MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query.


> Multi query retirver: Simple similarity search may be ambiguous. Confusion in the concept.
    > Tries to remove ambiguity from the users query. 
> Contexual Compression retriever

>>>FAISS - A vector store from facebook.

In [24]:
import os
os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [25]:
from langchain_community.retrievers import WikipediaRetriever

In [26]:
# Initialize the retriever (set language and top_k results)
retriever = WikipediaRetriever( lang="en", top_k_results=2)

In [27]:
# Query defination
query = "What is the capital of France?"
# Retrieve relevant Wikipedia articles
res = retriever.invoke(query)

In [28]:
for i, doc in enumerate(res):
    print(f"Document {i+1}:")
    print(f"Title: {doc.metadata['title']}")
    print(f"Content: {doc.page_content[:500]}...")  # Print the first 500 characters
    print("\n---\n")

Document 1:
Title: Closed-ended question
Content: A closed-ended question is any question for which a researcher provides research participants with options from which to choose a response. Closed-ended questions are sometimes phrased as a statement that requires a response.
A closed-ended question contrasts with an open-ended question, which cannot easily be answered with specific information.


== Examples ==
Examples of closed-ended questions that may elicit a "yes" or "no" response include:

Were you born in 2010?
Is Lyon the capital of Fra...

---

Document 2:
Title: France
Content: France, officially the French Republic, is a country primarily located in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the North Atlantic, the French West Indies, and many islands in Oceania and the Indian Ocean. Metropolitan France shares borders with Belgium and Luxembourg to the north; Germany to the northeast; Switzerland 

Vector Store Retriever

In [29]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface  import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [30]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [31]:
# Step 2: Initialize embedding model
embedding_model = HuggingFaceEmbeddings()

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [33]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [34]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Chroma is a vector database optimized for LLM-based search.


Maximal Marginal Relevance

In [35]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [36]:
from langchain_community.vectorstores import FAISS

# Initialize Huggingface embeddings
embedding_model = HuggingFaceEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = top results, lambda_mult = relevance-diversity balance
)

In [38]:
query = "What is langchain?"
results = retriever.invoke(query)

In [39]:
for i, doc in enumerate(results):
    print(f"\n---------results {i+1}---------")
    print(doc.page_content)


---------results 1---------
LangChain is used to build LLM based applications.

---------results 2---------
LangChain supports Chroma, FAISS, Pinecone, and more.

---------results 3---------
LangChain makes it easy to work with LLMs.


Multi Query Retirever